### 1. Import Dependecies

In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
                                    KFold, 
                                    GridSearchCV
                                    )
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')

### 2. Load the data

In [2]:
X_train = pd.read_csv('artifacts/X_train.csv')
Y_train = pd.read_csv('artifacts/y_train.csv')
X_test = pd.read_csv("artifacts/X_test.csv")
Y_test = pd.read_csv("artifacts/y_test.csv")

### 3. Define Paramters

In [3]:
lr_param_grid = {}

xgb_param_grid = {
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.05, 0.1, 0.2],
    'model__n_estimators': [100, 200]
}

rf_param_grid = {
    'model__n_estimators' : [100, 200],
    'model__max_depth' : [8, 10, 12],
    'model__min_samples_leaf': [2, 5, 10]
}

param_grids = {
    'Linear Regression': lr_param_grid,
    'XGBoost': xgb_param_grid,
    'Random Forest': rf_param_grid
}

### 4. Define Multi Models

In [4]:
models = {
        'Linear Regression' : LinearRegression(),
        'XGBoost' : XGBRegressor(random_state=42),
        'Random Forest' : RandomForestRegressor(random_state=42)
        }

### 5. Configure K-Fold CV

In [5]:
cv = KFold(
            n_splits=6,
            random_state=42,
            shuffle=True
            )

### 6. Multi Model Training

In [6]:
grid_search_results={}
for model_name, model in models.items():

    print(f"\n--- Tuning {model_name} ---")

    # Create Pipeline
    pipeline = Pipeline([
        ('model', model)
    ])

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
                                estimator=pipeline,
                                param_grid=param_grid,
                                cv=cv, scoring='r2',
                                verbose=1, return_train_score=False,
                                n_jobs=-1
                                )
    
    print(f"Fitting gridSearchCV for {model_name}")

    grid_search.fit(X_train, Y_train.values.ravel())

    grid_search_results[model_name] = grid_search
    
    print(f"{model_name} gridSearchCV completed ...")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")


--- Tuning Linear Regression ---
Fitting gridSearchCV for Linear Regression
Fitting 6 folds for each of 1 candidates, totalling 6 fits
Linear Regression gridSearchCV completed ...
Best parameters: {}
Best CV score: 0.6535

--- Tuning XGBoost ---
Fitting gridSearchCV for XGBoost
Fitting 6 folds for each of 18 candidates, totalling 108 fits
XGBoost gridSearchCV completed ...
Best parameters: {'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__n_estimators': 200}
Best CV score: 0.8300

--- Tuning Random Forest ---
Fitting gridSearchCV for Random Forest
Fitting 6 folds for each of 18 candidates, totalling 108 fits
Random Forest gridSearchCV completed ...
Best parameters: {'model__max_depth': 12, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
Best CV score: 0.7878


In [7]:
grid_search_results

{'Linear Regression': GridSearchCV(cv=KFold(n_splits=6, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('model', LinearRegression())]), n_jobs=-1,
              param_grid={}, scoring='r2', verbose=1),
 'XGBoost': GridSearchCV(cv=KFold(n_splits=6, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('model', XGBRegressor(random_state=42))]),
              n_jobs=-1,
              param_grid={'model__learning_rate': [0.05, 0.1, 0.2],
                          'model__max_depth': [4, 6, 8],
                          'model__n_estimators': [100, 200]},
              scoring='r2', verbose=1),
 'Random Forest': GridSearchCV(cv=KFold(n_splits=6, random_state=42, shuffle=True),
              estimator=Pipeline(steps=[('model', RandomForestRegressor(random_state=42))]),
              n_jobs=-1,
              param_grid={'model__max_depth': [8, 10, 12],
                          'model__min_samples_leaf': [2, 5, 10],
                          'model_

In [8]:
for model_name, grid in grid_search_results.items():
    # Get the best estimator from GridSearchCV
    best_model = grid.best_estimator_
    
    # Make predictions on the test set
    y_pred = best_model.predict(X_test)
    
    # Compute metrics
    mse = mean_squared_error(Y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(Y_test, y_pred)
    r2 = r2_score(Y_test, y_pred)
    
    print(f"\n--- {model_name} Test Set Metrics ---")
    print(f"MSE:       {mse:.4f}")
    print(f"RMSE:      {rmse:.4f}")
    print(f"MAE:       {mae:.4f}")
    print(f"R2-score:  {r2:.4f}")


--- Linear Regression Test Set Metrics ---
MSE:       0.4480
RMSE:      0.6693
MAE:       0.4805
R2-score:  0.6521

--- XGBoost Test Set Metrics ---
MSE:       0.2246
RMSE:      0.4739
MAE:       0.3124
R2-score:  0.8256

--- Random Forest Test Set Metrics ---
MSE:       0.2764
RMSE:      0.5257
MAE:       0.3522
R2-score:  0.7854
